<a href="https://colab.research.google.com/github/D2718281828nis/AutomaticControlTheory-Denoising_Signals/blob/main/lesson-4_MCP_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import inspect
from typing import Callable, Any, List, Tuple, Annotated, TypedDict, Literal
from openai import OpenAI

### How the Code Works

This notebook demonstrates how an AI assistant can use tools to respond to user queries. The core logic involves:

1.  **`build_initial_prompt`**: This function constructs a prompt for the Large Language Model (LLM), including a description of available tools and the user's query.
2.  **`call_llm`**: This function sends the prompt to the LLM (OpenAI's `gpt-5-nano` in this case) and retrieves its response.
3.  **`run_agent_inner`**: This is the main orchestrator. It parses the LLM's output. If the LLM decides to use a tool, it calls `process_tool_call`. If the LLM provides a final answer, it returns that answer.
4.  **`process_tool_call`**: This function executes the specified tool with the arguments provided by the LLM, and the tool's result is then fed back to the LLM in a subsequent prompt.

This creates a loop where the LLM can decide to use a tool, receive the tool's output, and then use that output to formulate a final answer.

### What is a Tool?

In this context, a **tool** is a specific function or capability that the AI assistant can call upon to perform a task or retrieve information that it cannot do intrinsically. For example, `calculate_bmi` (and later `bmi_tool` and `langchain_bmi`) is a tool that calculates the Body Mass Index based on weight and height.

Tools are defined with:
*   A **name**: For the LLM to identify it.
*   A **description**: To explain its purpose to the LLM.
*   **Arguments**: The inputs the tool requires.
*   **Outputs/Returns**: The type of information the tool will provide back.

### What is Wrapping?

**Wrapping** refers to the process of standardizing and augmenting the definition of a tool. Instead of manually writing a description for each tool (like in `RAW_TOOLS_DESC`), wrapping automates this process.

In this notebook, the `Tool` class and the `wrap_tool` decorator are used for wrapping:

*   The `Tool` class provides a consistent structure to store a tool's name, description, function reference, arguments, and return types.
*   The `wrap_tool` decorator (e.g., used with `bmi_tool`) automatically extracts this metadata (like argument names, types, and the function's docstring for description) from a Python function using introspection (`inspect` module).

This makes it easier to manage and describe tools, especially when dealing with many of them, ensuring they are presented to the LLM in a uniform and clear format.

# 0. Environment setup

In [13]:
from google.colab import userdata
import time
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "gpt-5.4-mini"  # Updated to a standard reliable model
client = OpenAI()

def call_llm(prompt: str) -> str:
    response = client.responses.create(model=MODEL, input=prompt)
    time.sleep(20) # Add a delay to avoid rate limiting
    return response.output_text

# 1. How to build tools

In [15]:
import re
import json

"""# 1. How to build tools
1.1. How tool-calling works
"""
def build_initial_prompt(tools_descr: str, user_text: str) -> str:
    system_prompt = f"""
You are a medical assistant with tool-calling capabilities.
You have access to the following tools:
{tools_descr}

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "tool_call", "name": "<tool_name>", "arguments": {{...}}}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "final", "answer": "..."}}
"""
    return f"{system_prompt}\n\nUSER: {user_text}\n"

def process_tool_call(json_model_output, tools_by_name):
    tool_name = json_model_output["name"]
    tool_args = json_model_output["arguments"]
    print(f'Executing tool {tool_name} with args: {tool_args}')
    if tool_name not in tools_by_name:
        raise ValueError("Unknown tool")

    tool_result = tools_by_name[tool_name](**tool_args)
    print(f'Tool result: {tool_result}')
    return tool_result

def run_agent_inner(prompt, tools_by_name):
    model_output = call_llm(prompt)
    print(f"Raw model_output from LLM: {model_output}") # Debugging

    json_model_output = None
    try:
        # Attempt direct parsing
        json_model_output = json.loads(model_output)
    except json.JSONDecodeError as e:
        print(f"Direct JSON parse failed: {e}. Attempting robust extraction.")

        # Try to extract the first valid JSON object using raw_decode
        decoder = json.JSONDecoder()
        try:
            # raw_decode can handle 'Extra data' and extract the first complete object
            json_model_output, _ = decoder.raw_decode(model_output.strip()) # strip to handle leading/trailing whitespace
            print(f"Extracted first JSON object using raw_decode: {json_model_output}")
        except json.JSONDecodeError as raw_decode_e:
            print(f"raw_decode also failed: {raw_decode_e}. Falling back to regex.")
            # If raw_decode fails, it means the start of the string isn't a valid JSON.
            # Try to find JSON within markdown blocks or as a standalone object using regex.

            # Try to find a JSON markdown block
            json_pattern_md = re.compile(r"```json\n(\{.*?})\n```", re.DOTALL)
            match_md = json_pattern_md.search(model_output)

            if match_md:
                json_str = match_md.group(1)
                print(f"Extracted JSON from markdown block: {json_str}")
                try:
                    json_model_output = json.loads(json_str)
                except json.JSONDecodeError as inner_e:
                    print(f"Error decoding extracted JSON from markdown: {inner_e}")
                    raise inner_e # Re-raise if markdown extraction fails to parse
            else:
                # If no markdown block, try to find a standalone JSON object using a more general regex.
                json_pattern_plain = re.compile(r"(\{.*?})", re.DOTALL)
                match_plain = json_pattern_plain.search(model_output)

                if match_plain:
                    json_str = match_plain.group(0) # Get the entire matched group
                    print(f"Extracted plain JSON string: {json_str}")
                    try:
                        json_model_output = json.loads(json_str)
                    except json.JSONDecodeError as inner_e:
                        print(f"Error decoding extracted plain JSON: {inner_e}")
                        raise inner_e # Re-raise if plain regex extraction fails to parse
                else:
                    print(f"Could not find any JSON object in model_output: {model_output}")
                    raise e # Re-raise the original (or latest) JSONDecodeError if no JSON could be extracted

    if json_model_output is None:
        raise ValueError("Failed to parse LLM output into a valid JSON object.")

    print(f"Parsed json_model_output: {json_model_output}") # Debugging

    if json_model_output["type"] == "tool_call":
        tool_result = process_tool_call(json_model_output, tools_by_name)
        prompt_with_tool_result = (
            f"{prompt}\n"
            f"ASSISTANT: {model_output}\n" # Keep original model_output here
            f"TOOL_RESULT: {tool_result}\n"
        )
        return run_agent_inner(prompt_with_tool_result, tools_by_name)

    print("\n\n==================Last prompt==================")
    print(prompt)
    print(f"\n\n====Final result: {json_model_output['answer']}=== ")
    return json_model_output["answer"]

"""## 1.2. Building hand-crafted tools (Medical Example)"""
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI)"""
    if height_m <= 0:
        return 0.0
    return round(weight_kg / (height_m ** 2), 2)

RAW_TOOLS_BY_NAME = {'calculate_bmi': calculate_bmi}
RAW_TOOLS_DESC = """
Tool name: calculate_bmi
Description: Calculate Body Mass Index (BMI) given weight in kilograms and height in meters.
Arguments:
- weight_kg: float
- height_m: float
Returns: float
"""

def run_agent_with_raw_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(RAW_TOOLS_DESC, user_text)
    return run_agent_inner(prompt, RAW_TOOLS_BY_NAME)

# 🩺 Medical examples instead of math
run_agent_with_raw_tools("What is the BMI for a patient weighing 70 kg and 1.75 m tall?")
run_agent_with_raw_tools("Calculate BMI for 85 kg weight and 1.80 m height.")



User query: What is the BMI for a patient weighing 70 kg and 1.75 m tall?


Raw model_output from LLM: {"type":"tool_call","name":"calculate_bmi","arguments":{"weight_kg":70,"height_m":1.75}}
Parsed json_model_output: {'type': 'tool_call', 'name': 'calculate_bmi', 'arguments': {'weight_kg': 70, 'height_m': 1.75}}
Executing tool calculate_bmi with args: {'weight_kg': 70, 'height_m': 1.75}
Tool result: 22.86
Raw model_output from LLM: {"type":"final","answer":"The patient's BMI is 22.86."}
Parsed json_model_output: {'type': 'final', 'answer': "The patient's BMI is 22.86."}


==================Last prompt==================

You are a medical assistant with tool-calling capabilities.
You have access to the following tools:

Tool name: calculate_bmi
Description: Calculate Body Mass Index (BMI) given weight in kilograms and height in meters.
Arguments:
- weight_kg: float
- height_m: float
Returns: float


When you need to use a tool, output EXACTLY one JSON object (no extra text) in this f

'Your BMI is 26.23.'

# 1.3. Wrapping tools

In [16]:
"""1.3. Wrapping tools
1.3.1. Using class for unification (Fixed syntax)"""
class Tool:
    def __init__(self, name: str, description: str, func: Callable[..., Any],
                 arguments: List[Tuple[str, str]], outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        args_str = ", ".join([f"{n}: {t}" for n, t in self.arguments])
        return (f"Tool Name: {self.name}, "
                f"Description: {self.description}, "
                f"Arguments: {args_str}, "
                f"Outputs: {self.outputs}")

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

"""1.3.2. "Decorating" tools (Fixed inspect._empty)"""
def wrap_tool(func: Callable[..., Any]) -> Tool:
    sig = inspect.signature(func)
    arguments = []
    for p in sig.parameters.values():
        ann = p.annotation
        if ann is inspect.Parameter.empty:
            ann_name = "Any"
        else:
            ann_name = getattr(ann, "__name__", str(ann))
        arguments.append((p.name, ann_name))

    ret = sig.return_annotation
    if ret is inspect.Parameter.empty:
        outputs = "Any"
    else:
        outputs = getattr(ret, "__name__", str(ret))

    description = (func.__doc__ or "No description provided").strip()
    name = func.__name__
    return Tool(name, description, func, arguments, outputs)

@wrap_tool
def bmi_tool(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI) given weight in kg and height in meters."""
    return calculate_bmi(weight_kg, height_m)

bmi_tool.to_string()

"""1.3.3. All together"""
NICE_TOOLS_BY_NAME = {'bmi_tool': bmi_tool}
NICE_TOOLS_DESC = "\n".join([t.to_string() for t in NICE_TOOLS_BY_NAME.values()])

def run_agent_with_nice_tools(user_text: str):
    print(f"\n\nUser query: {user_text}\n\n")
    prompt = build_initial_prompt(NICE_TOOLS_DESC, user_text)
    return run_agent_inner(prompt, NICE_TOOLS_BY_NAME)

run_agent_with_nice_tools("What is the BMI for 65 kg and 1.70 m?")



User query: What is the BMI for 65 kg and 1.70 m?


Raw model_output from LLM: {"type":"tool_call","name":"bmi_tool","arguments":{"weight_kg":65,"height_m":1.7}}
Parsed json_model_output: {'type': 'tool_call', 'name': 'bmi_tool', 'arguments': {'weight_kg': 65, 'height_m': 1.7}}
Executing tool bmi_tool with args: {'weight_kg': 65, 'height_m': 1.7}
Tool result: 22.49
Raw model_output from LLM: {"type":"final","answer":"The BMI for 65 kg and 1.70 m is 22.49."}
Parsed json_model_output: {'type': 'final', 'answer': 'The BMI for 65 kg and 1.70 m is 22.49.'}


==================Last prompt==================

You are a medical assistant with tool-calling capabilities.
You have access to the following tools:
Tool Name: bmi_tool, Description: Calculate Body Mass Index (BMI) given weight in kg and height in meters., Arguments: weight_kg: float, height_m: float, Outputs: float

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{"type": "tool_call", "name

'The BMI for 65 kg and 1.70 m is 22.49.'

The run_agent_inner function orchestrates the interaction with the LLM and the tools. Here's a breakdown of how it handles tool loops:

* Initial LLM Call: It first calls the LLM with the current prompt.
Parse LLM Output: The response from the LLM is expected to be a JSON object. This JSON is parsed to determine if the LLM decided to make a tool_call or provide a final answer.
* Tool Call Detection: If json_model_output["type"] == "tool_call":
It extracts the tool_name and arguments from the LLM's output.
It then calls process_tool_call with these details, which executes the specified tool.
* The result of the tool execution (tool_result) is then appended to the prompt, along with the LLM's initial tool call message.
Crucially, run_agent_inner recursively calls itself with this updated prompt. This creates the 'tool loop', allowing the LLM to process the tool's output and potentially make further decisions (another tool call or a final answer).
Final Answer Detection: If json_model_output["type"] == "final":

The function extracts the answer from the LLM's output.
It then prints the final prompt and the answer, and returns the final answer, terminating the loop.
In essence, run_agent_inner acts as a state machine: it continuously interacts with the LLM, executing tools as instructed, until the LLM provides a final answer.

# 1.4. Using libraries (LangChain + LangGraph)

In [17]:
"""## 1.4. Using libraries (LangChain + LangGraph)"""
!pip install langgraph langchain langchain-openai -q
from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 38.4 MB/s eta 0:00:00


In [18]:

@tool
def langchain_bmi(weight_kg: float, height_m: float) -> float:
    """Calculate Body Mass Index (BMI)."""
    return calculate_bmi(weight_kg, height_m)

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

llm = ChatOpenAI(model=MODEL, temperature=0).bind_tools([langchain_bmi])

def llm_node(state: AgentState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

tools_node = ToolNode([langchain_bmi])

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tools_node)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")
app = graph.compile()

# 🩺 Medical test with LangGraph
result = app.invoke({
    "messages": [
        SystemMessage(content="You are a medical assistant. Always try to use provided tools."),
        HumanMessage(content="What is the BMI for a patient weighing 90 kg and 1.85 m tall?")
    ]
})
print(result["messages"][-1].content)

# Streaming demo
for step in app.stream(
    {
        "messages": [
            SystemMessage(content="You are a medical assistant. Always try to use provided tools."),
            HumanMessage(content="Calculate BMI for 75 kg and 1.78 m.")
        ]
    },
    stream_mode="values",
):
    last = step["messages"][-1]
    print("NODE OUTPUT:")
    print(type(last).__name__)
    if hasattr(last, "content"):
        print("Content:", last.content)
    if hasattr(last, "tool_calls") and last.tool_calls:
        print("Tool calls:", last.tool_calls)
    print("==========")

The BMI is **26.3 kg/m²**.
NODE OUTPUT:
HumanMessage
Content: Calculate BMI for 75 kg and 1.78 m.
NODE OUTPUT:
AIMessage
Content: 
Tool calls: [{'name': 'langchain_bmi', 'args': {'weight_kg': 75, 'height_m': 1.78}, 'id': 'call_TG1GbnV1XDxTzYEDRHMlKGui', 'type': 'tool_call'}]
NODE OUTPUT:
ToolMessage
Content: 23.67
NODE OUTPUT:
AIMessage
Content: Your BMI is **23.7** (rounded to one decimal place).


# MCP fundamentals

In [20]:
import os
import json
import inspect
import time
from typing import Callable, Any, List, Tuple
from openai import OpenAI
from google.colab import userdata
import re

# Re-using Tool and wrap_tool from previous cells (12oZG_YcvRoF)
class Tool:
    def __init__(self, name: str, description: str, func: Callable[..., Any],
                 arguments: List[Tuple[str, str]], outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        args_str = ", ".join([f"{n}: {t}" for n, t in self.arguments])
        return (f"Tool Name: {self.name}, "
                f"Description: {self.description}, "
                f"Arguments: {args_str}, "
                f"Outputs: {self.outputs}")

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

def wrap_tool(func: Callable[..., Any]) -> Tool:
    sig = inspect.signature(func)
    arguments = []
    for p in sig.parameters.values():
        ann = p.annotation
        if ann is inspect.Parameter.empty:
            ann_name = "Any"
        else:
            ann_name = getattr(ann, "__name__", str(ann))
        arguments.append((p.name, ann_name))

    ret = sig.return_annotation
    if ret is inspect.Parameter.empty:
        outputs = "Any"
    else:
        outputs = getattr(ret, "__name__", str(ret))

    description = (func.__doc__ or "No description provided").strip()
    name = func.__name__
    return Tool(name, description, func, arguments, outputs)


# 1. Define a simple tool: `add_numbers`
@wrap_tool
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers and returns their sum."""
    return a + b

# 2. Set up tools for the agent for this specific example
MCP_TOOLS_BY_NAME = {'add_numbers': add_numbers}
MCP_TOOLS_DESC = "\n".join([t.to_string() for t in MCP_TOOLS_BY_NAME.values()])

# Re-using (and redefining for self-containment) build_initial_prompt and call_llm
# as they are central to the protocol demonstration.
def build_initial_prompt(tools_descr: str, user_text: str) -> str:
    system_prompt = f"""
You are a helpful assistant with tool-calling capabilities.
You have access to the following tools:
{tools_descr}

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "tool_call", "name": "<tool_name>", "arguments": {{...}}}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{{"type": "final", "answer": "..."}}
"""
    return f"{system_prompt}\n\nUSER: {user_text}\n"

# Assuming OPENAI_API_KEY and MODEL are set globally from previous cells.
# Redefining for self-containment, but using the global client if available.
try:
    client
except NameError:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    MODEL = "gpt-5.4-mini"
    client = OpenAI()

def call_llm(prompt: str) -> str:
    print(f"  [DEBUG] Calling LLM with prompt segment:\n{prompt[:200]}...")
    response = client.responses.create(model=MODEL, input=prompt)
    time.sleep(15) # Add a delay to avoid rate limiting for demonstration
    return response.output_text


# Redefine process_tool_call with additional prints for visualization
def process_tool_call_mcp(json_model_output, tools_by_name, indent=""):
    tool_name = json_model_output["name"]
    tool_args = json_model_output["arguments"]
    print(f"{indent}  ⚙️ Executing tool '{tool_name}' with args: {tool_args}")
    if tool_name not in tools_by_name:
        raise ValueError("Unknown tool")
    tool_result = tools_by_name[tool_name](**tool_args)
    print(f"{indent}  Tool '{tool_name}' returned: {tool_result}")
    return tool_result

# 3. Enhanced `run_agent_inner` for visualization of Model Context Protocol
def run_mcp_agent_inner(prompt: str, tools_by_name: dict, depth: int = 0):
    indent = "  " * depth
    print(f"\n{indent}╔═══════════════════════════════════════════════╗")
    print(f"{indent}║  LLM Interaction Cycle (Depth: {depth}) ║")
    print(f"{indent}╚═══════════════════════════════════════════════╝")

    print(f"{indent}Prompt sent to LLM:\n{indent}--------------------\n{prompt}\n{indent}--------------------")

    model_output = call_llm(prompt)
    print(f"{indent}LLM Raw Output: {model_output}")

    json_model_output = None
    try:
        json_model_output = json.loads(model_output)
    except json.JSONDecodeError as e:
        print(f"{indent}   JSON parse failed directly: {e}. Attempting robust extraction.")
        decoder = json.JSONDecoder()
        try:
            json_model_output, _ = decoder.raw_decode(model_output.strip())
        except json.JSONDecodeError as raw_decode_e:
            json_pattern_md = re.compile(r"```json\n(\{.*?})\n```", re.DOTALL)
            match_md = json_pattern_md.search(model_output)
            if match_md:
                json_str = match_md.group(1)
                json_model_output = json.loads(json_str)
            else:
                json_pattern_plain = re.compile(r"(\{.*?})", re.DOTALL)
                match_plain = json_pattern_plain.search(model_output)
                if match_plain:
                    json_str = match_plain.group(0)
                    json_model_output = json.loads(json_str)
                else:
                    raise e # Re-raise if no JSON could be extracted after all attempts
    print(f"{indent}LLM Parsed Output: {json_model_output}")

    if json_model_output["type"] == "tool_call":
        print(f"{indent} LLM decided to call a tool.")
        tool_result = process_tool_call_mcp(json_model_output, tools_by_name, indent)

        # Key aspect of MCP: Tool result is added to the context (prompt) for the next LLM call
        prompt_with_tool_result = (
            f"{prompt}\n"
            f"ASSISTANT: {model_output}\n" # Include LLM's tool call as part of history
            f"TOOL_RESULT: {tool_result}\n"
        )
        print(f"{indent} Tool result added to context. Re-prompting LLM for next step (recursive call)...")
        return run_mcp_agent_inner(prompt_with_tool_result, tools_by_name, depth + 1)
    elif json_model_output["type"] == "final":
        print(f"{indent} LLM provided a final answer. Conversation ends.")
        print(f"{indent}-------------------------------------------------")
        print(f"{indent}Final Answer: {json_model_output['answer']}")
        print(f"{indent}-------------------------------------------------")
        return json_model_output["answer"]
    else:
        raise ValueError(f"{indent}Unexpected LLM output type: {json_model_output['type']}")

# 4. Function to run the MCP demonstration
def run_mcp_demonstration(user_text: str):
    print(f"=================================================")
    print(f"      Model Context Protocol Demonstration ✨   ")
    print(f"=================================================")
    print(f"User Query: \"{user_text}\"\n")
    initial_prompt = build_initial_prompt(MCP_TOOLS_DESC, user_text)
    return run_mcp_agent_inner(initial_prompt, MCP_TOOLS_BY_NAME)

# 5. Example usage: Demonstrating a query that requires a tool call
run_mcp_demonstration("What is the sum of 12 and 34?")

      Model Context Protocol Demonstration ✨   
User Query: "What is the sum of 12 and 34?"


╔═══════════════════════════════════════════════╗
║  LLM Interaction Cycle (Depth: 0) ║
╚═══════════════════════════════════════════════╝
Prompt sent to LLM:
--------------------

You are a helpful assistant with tool-calling capabilities.
You have access to the following tools:
Tool Name: add_numbers, Description: Adds two numbers and returns their sum., Arguments: a: float, b: float, Outputs: float

When you need to use a tool, output EXACTLY one JSON object (no extra text) in this format:
{"type": "tool_call", "name": "<tool_name>", "arguments": {...}}

When you want to answer the user, output EXACTLY one JSON object (no extra text) in this format:
{"type": "final", "answer": "..."}


USER: What is the sum of 12 and 34?

--------------------
  [DEBUG] Calling LLM with prompt segment:

You are a helpful assistant with tool-calling capabilities.
You have access to the following tools:
Tool Na

'46'